# 12 -- Threshold Optimisation
**Purpose:** Find the optimal decision threshold for the champion model that maximises fraud detection while controlling false positives.

**Input:** Champion model + X_test, y_test
**Output:** `config/threshold.yaml` | `reports/12_threshold_analysis.txt`

## Learning Objectives
- Understand why 0.5 is almost never the right threshold for fraud
- Build a precision-recall curve and interpret it
- Select a threshold based on a business cost criterion
- Understand how threshold affects risk tier distribution (GREEN / AMBER / RED)

## Business Context
CrossBorderPay blocks payments in the RED tier and reviews AMBER. Every false positive (legitimate
transaction blocked) has a cost: the customer's business is disrupted and they may churn.
Every false negative (fraud missed) has a cost: financial loss and regulatory risk.
The optimal threshold minimises the total cost -- not just one side.

## Step 1 -- Plot Precision-Recall Curve
Plot precision and recall as a function of threshold (0.0 to 1.0).
Mark the F1-maximising threshold. Mark the recall=75% threshold. Mark 0.5.

## Step 2 -- Business Cost Analysis
Assign: FP cost = $500 (investigation + potential customer churn). FN cost = $25,000 (average fraud loss).
At each threshold, compute: total_cost = (FP_count * 500) + (FN_count * 25000).
Which threshold minimises total cost?

## Step 3 -- Risk Tier Distribution
With the chosen threshold, map scores to GREEN / AMBER / RED tiers.
Print: how many transactions fall in each tier? What percentage of fraud falls in RED?

## Definition of Done
- [ ] Precision-recall curve produced
- [ ] Business cost analysis completed
- [ ] Optimal threshold selected with written justification
- [ ] Risk tier distribution printed
- [ ] Threshold saved to config/threshold.yaml

In [1]:
import pandas as pd
import joblib
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score

X_test  = pd.read_parquet(r"../data/processed\X_test.parquet")
y_test  = pd.read_parquet(r"../data/processed\y_test.parquet").squeeze()
model   = joblib.load(r"../data/processed\xgboost_champion.pkl")

y_prob = model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Find best threshold: highest precision where recall >= 0.75
mask = recall[:-1] >= 0.75
best_idx = np.argmax(precision[:-1][mask])
best_threshold = thresholds[mask][best_idx]
best_precision = precision[:-1][mask][best_idx]
best_recall    = recall[:-1][mask][best_idx]

print(f"Optimal threshold:  {best_threshold:.4f}")
print(f"Precision at that:  {best_precision:.2%}")
print(f"Recall at that:     {best_recall:.2%}")


Optimal threshold:  0.9217
Precision at that:  100.00%
Recall at that:     82.00%


In [4]:
import json

threshold_config = {
    "model": "XGBoost",
    "optimal_threshold": round(float(best_threshold), 4),
    "precision_at_threshold": 1.0,
    "recall_at_threshold": 0.82,
    "false_positives_at_threshold": 0,
    "false_negatives_at_threshold": 9,
    "selection_criteria": "max precision subject to recall >= 0.75"
}

with open(r"../data/processed\threshold_config.json", "w") as f:
    json.dump(threshold_config, f, indent=2)

print(f"Saved. Production threshold: {best_threshold:.4f}")


Saved. Production threshold: 0.9217
